In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [ ]:
#读取行业信息数据
stock_industry = pd.read_excel(r"E:\A股leverage研究\raw data\data_info.xlsx",skiprows=[0,2]) #去掉英文代码以及单位信息

#将证券代码填充到六位
stock_industry['证券代码'] = stock_industry['证券代码'].apply(lambda x: str(x).zfill(6)) #经检查无重复的证券代码

#选择需要的列
stock_industry = stock_industry[['证券代码','证券简称','上市日期','行业代码D','公司活动情况']]



In [ ]:
#读取股票收益率数据
stock_returns = pd.read_excel(r"E:\A股leverage研究\raw data\data_return.xlsx",skiprows=[0,2]) #去掉英文代码以及单位信息

#将证券代码填充到六位
stock_returns['证券代码'] = stock_returns['证券代码'].apply(lambda x: str(x).zfill(6)).astype('str') 

#将交易月份转换为datetime格式
stock_returns['交易月份'] = pd.to_datetime(stock_returns['交易月份'], format='%Y-%m')

# #将交易月份对齐到月初
# stock_returns['交易月份'] = stock_returns['交易月份']- pd.offsets.MonthBegin(1)

#选择需要的列
stock_returns = stock_returns[['证券代码','交易月份','月收盘价','月个股流通市值','月个股总市值','考虑现金红利再投资的月个股回报率','市场类型']]

#重命名部分列
stock_returns.rename(columns={'考虑现金红利再投资的月个股回报率':'月个股回报率'}, inplace=True)

#将月个股流通市值与月个股总市值的单位从千元换算成元
stock_returns['月个股流通市值'] = stock_returns['月个股流通市值'] * 1000
stock_returns['月个股总市值'] = stock_returns['月个股总市值'] * 1000

#对市场类型筛选，只取所有的A股（1：主板A股，4：深证A股，64：北证A股）
stock_returns  = stock_returns.astype({'市场类型': 'str'}).query("市场类型 in ['1','4','16','32','64']").reset_index(drop=True)


#添加decdate列，用于与财务数据合并
stock_returns = stock_returns.assign(
    DecDate=lambda x: pd.to_datetime({
        'year': x['交易月份'].dt.year - 1 - (x['交易月份'].dt.month < 7).astype(int),
        'month': 12,
        'day': 1
    })
)

In [ ]:
stock_returns

In [ ]:
stock_returns.duplicated().sum()  #检查是否有重复行，结果为0

In [ ]:
#取出月个股市值
stock_me = stock_returns[['证券代码','交易月份','月个股总市值','月个股流通市值']].copy()

#六月市值
stock_me_jun = (
    stock_me.query('交易月份.dt.month==6')
    .assign(交易月份 = lambda x: pd.to_datetime((x['交易月份'].dt.year-1).astype(str) + '-12-01'))
    .loc[:, ['证券代码','交易月份','月个股总市值']]
    .rename(columns={'交易月份': 'DecDate','月个股总市值': 'mejun'})
)

#十二月市值
stock_me_dec = (
    stock_me.query('交易月份.dt.month==12')
    .assign(交易月份 = lambda x: pd.to_datetime(x['交易月份'].dt.year.astype(str) + '-12-01'))
    .loc[:, ['证券代码','交易月份','月个股总市值']]
    .rename(columns={'交易月份': 'DecDate','月个股总市值': 'medec'})
)

In [ ]:
stock_me_jun

In [ ]:
stock_me_dec

In [ ]:
#读取财务数据
stock_finance = pd.read_excel(r"E:\A股leverage研究\raw data\data_financial.xlsx",skiprows=[0,2]) #去掉英文代码以及单位信息

#将证券代码填充到六位
stock_finance['证券代码'] = stock_finance['证券代码'].apply(lambda x: str(x).zfill(6)) 

#将交易月份转换为datetime格式
stock_finance['统计截止日期'] = pd.to_datetime(stock_finance['统计截止日期'])

#筛选报表类型为合并报表（type为A）的数据
stock_finance = stock_finance[stock_finance['报表类型'] == 'A'].reset_index(drop=True)

#将优先股的NaN值填充为0
stock_finance['其中：优先股'] = stock_finance['其中：优先股'].fillna(0)

#删除统计截至日期为1月初的数据
stock_finance = stock_finance[stock_finance.统计截止日期.dt.month != 1].reset_index(drop=True)

#计算账面价值（book equity = 所有者权益合计-优先股）
stock_finance ['账面价值'] = stock_finance['归属于母公司所有者权益合计'] - stock_finance['其中：优先股']
stock_finance.loc[stock_finance['账面价值'] < 0, '账面价值'] = np.nan


In [ ]:
stock_finance

In [ ]:
#将统计截止日期调整为月初日期
stock_finance['统计截止日期'] = stock_finance['统计截止日期'] - pd.offsets.MonthBegin(1)

In [ ]:
stock_finance

In [ ]:
#取出资产总计(total assets)和账面价值(book equity)两列
stock_at_be = stock_finance[['证券代码','统计截止日期','资产总计','账面价值']]

######be和at在每个财年保持不变，这里参考美股的做法，取年度的at和be数据######
#增加年份列
stock_at_be['年份'] = stock_at_be['统计截止日期'].dt.year

#取年报数据
stock_at_be = (stock_at_be
    .sort_values(by=['证券代码','统计截止日期'])
    .groupby(['证券代码','年份'])
    .apply(lambda x: x[x['统计截止日期'].dt.month == 12])
    .reset_index(drop=True)
)

#将统计截止日期改写成对应年度的十二月初的数据，用于与收益率数据合并
stock_at_be = stock_at_be.rename(columns={'统计截止日期':'DecDate'})

#删除年份列
stock_at_be = stock_at_be.drop(columns=['年份'])



In [ ]:
# ######lt需要季度的数据######
# ###填充版
# #取出负债合计(total liabilities)
# stock_lt = stock_finance[['证券代码','统计截止日期','负债合计']]

# #对数据进行扩充
# def fill_quarterly_data(df):
#     """
#     填充季度数据
    
#     参数:
#     df: DataFrame，包含三列：证券代码、统计截止日期、负债合计
    
#     返回:
#     填充后的DataFrame
#     """
#     # 获取列名
#     col_code = df.columns[0]  # 证券代码
#     col_date = df.columns[1]  # 统计截止日期
#     col_debt = df.columns[2]  # 负债合计
    
#     result_list = []
    
#     # 按证券代码分组处理
#     for code, group in df.groupby(col_code):
#         group = group.copy().sort_values(by=col_date)
        
#         # 获取该证券的所有年份
#         years = group[col_date].dt.year.unique()
        
#         for year in sorted(years):
#             # 获取该年度的数据
#             year_data = group[group[col_date].dt.year == year].copy()
#             year_data['month_day'] = year_data[col_date].dt.strftime('%m-%d')
            
#             # 获取该年度已有的日期
#             existing_dates = set(year_data['month_day'].tolist())
            
#             # 定义标准的四个季度
#             q1 = '03-01'
#             q2 = '06-01'
#             q3 = '09-01'
#             q4 = '12-01'
            
#             # 判断是否需要填充
#             # 如果已经有全部四个季度的数据，则不需要填充
#             has_all_quarters = {q1, q2, q3, q4}.issubset(existing_dates)
            
#             if not has_all_quarters:
#                 # 需要填充
#                 # Q1 (3.01) 用 Q2 (6.01) 的数据填充
#                 if q1 not in existing_dates and q2 in existing_dates:
#                     if year > 1994:  # 1994年不需要补充3.01
#                         source_row = year_data[year_data['month_day'] == q2].iloc[0]
#                         new_row = source_row.copy()
#                         new_row[col_date] = pd.to_datetime(f'{year}-{q1}')
#                         new_row['month_day'] = q1
#                         result_list.append(new_row)
                
#                 # Q3 (9.01) 用 Q4 (12.01) 的数据填充
#                 if q3 not in existing_dates and q4 in existing_dates:
#                     if year < 2025:  # 2025年无需补充，但9.01应该已经存在
#                         source_row = year_data[year_data['month_day'] == q4].iloc[0]
#                         new_row = source_row.copy()
#                         new_row[col_date] = pd.to_datetime(f'{year}-{q3}')
#                         new_row['month_day'] = q3
#                         result_list.append(new_row)
#                     # elif year == 1994:  # 1994年特殊处理：补充9.01
#                     #     if q4 in existing_dates:
#                     #         source_row = year_data[year_data['month_day'] == q4].iloc[0]
#                     #         new_row = source_row.copy()
#                     #         new_row[col_date] = pd.to_datetime(f'{year}-{q3}')
#                     #         new_row['month_day'] = q3
#                     #         result_list.append(new_row)
            
#             # 添加原有数据
#             for _, row in year_data.iterrows():
#                 result_list.append(row)
    
#     # 转换为DataFrame
#     result_df = pd.DataFrame(result_list)
    
#     # 删除辅助列
#     if 'month_day' in result_df.columns:
#         result_df = result_df.drop(columns=['month_day'])
    
#     # 去重并排序
#     result_df = result_df.drop_duplicates(subset=[col_code, col_date])
#     result_df = result_df.sort_values(by=[col_code, col_date])
#     result_df = result_df.reset_index(drop=True)
    
#     return result_df

# stock_lt_cleaned = fill_quarterly_data(stock_lt)
# stock_lt_cleaned = stock_lt_cleaned.rename(columns={'统计截止日期':'id'})

In [ ]:
# #####lt需要季度的数据######
# ## 非填充版
# #取出负债合计(total liabilities)
# stock_lt = stock_finance[['证券代码','统计截止日期','负债合计']]

# def build_quarterly_lt(df):
#     col_code = df.columns[0]
#     col_date = df.columns[1]
#     col_value = df.columns[2]
#     df_sorted = df.sort_values(by=[col_code, col_date]).reset_index(drop=True)

#     placeholder_rows = []
#     for code, group in df_sorted.groupby(col_code):
#         for year, year_data in group.groupby(group[col_date].dt.year):
#             existing_months = set(year_data[col_date].dt.month)
#             for month in (3, 9): 
#                 if month not in existing_months:
#                     placeholder_rows.append({
#                         col_code: code,
#                         col_date: pd.Timestamp(year=int(year), month=month, day=1),
#                         col_value: np.nan
#                     })

#     if placeholder_rows:
#         placeholder_df = pd.DataFrame(placeholder_rows)
#         result_df = pd.concat([df_sorted, placeholder_df], ignore_index=True)
#     else:
#         result_df = df_sorted.copy()

#     result_df = (result_df
#         .drop_duplicates(subset=[col_code, col_date], keep='first')
#         .sort_values(by=[col_code, col_date])
#         .reset_index(drop=True))
#     result_df = result_df.rename(columns={col_date: 'id'})
#     return result_df

# stock_lt_cleaned = build_quarterly_lt(stock_lt)


In [ ]:
# #####lt需要季度的数据######
# ## 前一季度填充版
# #取出负债合计(total liabilities)
stock_lt = stock_finance[['证券代码','统计截止日期','负债合计']]
def build_quarterly_lt_fill_next(df):
    col_code = df.columns[0]
    col_date = df.columns[1]
    col_value = df.columns[2]

    df_sorted = df.sort_values(by=[col_code, col_date]).reset_index(drop=True)

    # 生成每个股票在每年 3/6/9/12 的占位行
    placeholder_rows = []
    for code, group in df_sorted.groupby(col_code):
        for year, year_data in group.groupby(group[col_date].dt.year):
            existing_months = set(year_data[col_date].dt.month)
            for month in (3, 6, 9, 12):
                if month not in existing_months:
                    placeholder_rows.append({
                        col_code: code,
                        col_date: pd.Timestamp(year=int(year), month=month, day=1),
                        col_value: np.nan
                    })

    if placeholder_rows:
        placeholder_df = pd.DataFrame(placeholder_rows)
        result_df = pd.concat([df_sorted, placeholder_df], ignore_index=True)
    else:
        result_df = df_sorted.copy()

    result_df = (result_df
        .drop_duplicates(subset=[col_code, col_date], keep='first')
        .sort_values(by=[col_code, col_date])
        .reset_index(drop=True))

    # 用上一季度值填充缺失（按股票分组）
    result_df[col_value] = result_df.groupby(col_code)[col_value].ffill()

    result_df = result_df.rename(columns={col_date: 'id'})
    return result_df
stock_lt_cleaned = build_quarterly_lt_fill_next(stock_lt)

In [ ]:
stock_lt_cleaned

In [ ]:
stock_returns

In [ ]:
#与收益率数据合并
df = pd.merge(stock_returns, stock_industry, how='left', on='证券代码')

# #删除上市后前6个月的数据
# df['月末日期'] = df['交易月份'] + pd.offsets.MonthEnd(0)
# df = df[df['上市日期'] <= (df['月末日期'] - pd.DateOffset(months=6))].reset_index(drop=True)
# df = df.drop(columns=['月末日期'])
df['交易月份'] = pd.to_datetime(df['交易月份'])
df['上市日期'] = pd.to_datetime(df['上市日期'])
trade_m = df['交易月份'].dt.to_period('M')
list_m = df['上市日期'].dt.to_period('M')

months_diff = (trade_m.dt.year - list_m.dt.year) * 12 + (trade_m.dt.month - list_m.dt.month)

df = df[months_diff >= 6].reset_index(drop=True)

#剔除金融行业
df = df[~df['行业代码D'].astype(str).str.startswith('J')].reset_index(drop=True)

#去掉ST股
df = df[~df['证券简称'].astype(str).str.contains('ST')].reset_index(drop=True)

#去掉PT股
df = df[~df['证券简称'].astype(str).str.contains('PT')].reset_index(drop=True)

# #去掉非正常经营公司
# df = df[df['公司活动情况'] == 'A'].reset_index(drop=True)

#合并me_jun
df = pd.merge(df, stock_me_jun, how='left', on=['证券代码','DecDate'])

#合并me_dec
df = pd.merge(df, stock_me_dec, how='left', on=['证券代码','DecDate'])

#合并at_be
df = pd.merge(df, stock_at_be, how='left', on=['证券代码','DecDate'])



In [ ]:
df['year'] = df['交易月份'].dt.year
df['month'] = df['交易月份'].dt.month
def gen_id(row):
    year = row['year']
    month = row['month']
    
    # 计算当前季度的月份范围
    if month in [1,2,3]:
        return  str(year-1)+'-12'+'-01'
    elif month in [4,5,6]:
        return str(year)+'-3'+'-01'
    elif month in [7,8,9]:
        return str(year)+'-6'+'-01'
    elif month in [10,11,12]:
        return str(year)+'-9'+'-01'

df = df.astype({'year':int,'month':int}).assign(id=lambda x: x.apply(gen_id, axis=1)).assign(id =lambda x: pd.to_datetime(x['id']))

#合并lt
df = pd.merge(df, stock_lt_cleaned, how='left', on=['证券代码','id'])

#删除辅助列
df = df.drop(columns=['year','month','id'])

In [ ]:
#读取无风险利率-数据来源为中央财经大学fama五因子数据
df_rf = pd.read_csv(r"E:\A股leverage研究\raw data\fivefactor_monthly.csv")

df_rf = df_rf[['trdmn','rf']].rename(columns={'trdmn':'交易月份'}).assign(交易月份 = lambda x: pd.to_datetime(x['交易月份'], format='%Y%m') - pd.offsets.MonthBegin(1))

df = pd.merge(df, df_rf, how='left', on='交易月份')

#计算超额收益率
df['ret_excess'] = df['月个股回报率'] - df['rf']

#计算账面市值比bm
df['bm'] = df['账面价值'] / df['medec']

#计算杠杆率leverage
df['lev'] = df['负债合计'] / (df['负债合计']+df['月个股总市值'])




In [ ]:
df.lev.value_counts(dropna=False)

In [ ]:
df

In [ ]:
#添加滞后一期的杠杆率
lev_lag = (df
    .assign(
        交易月份=lambda x: x["交易月份"] + pd.DateOffset(months=1),
        lev_lag=lambda x: x["lev"]
    )
    .get(["证券代码", "交易月份", "lev_lag"])
)
# 将滞后杠杆率合并回主表
df = df.merge(lev_lag, how="left", on=["证券代码", "交易月份"])

In [ ]:
df.lev_lag.value_counts(dropna=False)   

In [ ]:
#添加滞后一期的总市值
me_lag_t = (df
    .assign(
        交易月份=lambda x: x["交易月份"] + pd.DateOffset(months=1),
        me_lag_t=lambda x: x["月个股总市值"]
    )
    .get(["证券代码", "交易月份", "me_lag_t"])
)
# 将滞后总市值合并回主表
df = df.merge(me_lag_t, how="left", on=["证券代码", "交易月份"])

In [ ]:
#添加滞后一期的流通市值
me_lag_f = (df
    .assign(
        交易月份=lambda x: x["交易月份"] + pd.DateOffset(months=1),
        me_lag_f=lambda x: x["月个股流通市值"]
    )
    .get(["证券代码", "交易月份", "me_lag_f"])
)
# 将滞后流通市值合并回主表
df = df.merge(me_lag_f, how="left", on=["证券代码", "交易月份"])

In [ ]:
# # 如果 date 是 datetime，其中 day 不重要，可以先转成年月
# df['year_month'] = df['交易月份'].dt.to_period('M')

# # 按月份分组，每月计算上月个股总市值的 30% 分位数
# quantile_30 = df.groupby('year_month')['me_lag_t'].quantile(0.3)

# # 将 quantile merge 回原表
# df = df.merge(quantile_30.rename('q30'), left_on='year_month', right_index=True)

# # 保留上月总市值 > 上月 30% 分位数的股票
# df = df[df['me_lag_t'] > df['q30']].copy()

# # 删除辅助列
# df.drop(columns=['q30', 'year_month'], inplace=True)

# # 1) 用上月为分组键（假设 交易月份 是当月）
# df['year_month_lag'] = (df['交易月份'] - pd.offsets.MonthBegin(1)).dt.to_period('M')

# # 2) 在“上月横截面”计算 30% 分位数
# quantile_30 = df.groupby('year_month_lag')['me_lag_t'].quantile(0.3)

# # 3) 把阈值合并回当月样本（按上月键）
# df = df.merge(quantile_30.rename('q30'), left_on='year_month_lag', right_index=True)

# # 4) 用上月阈值筛选当月样本
# df = df[df['me_lag_t'] > df['q30']].copy()

# # 5) 清理辅助列
# df.drop(columns=['q30', 'year_month_lag'], inplace=True)

In [ ]:
df = df[df['交易月份'] >= '1994-07-01'].reset_index(drop=True)
# df = df[df['交易月份'] >= '2000-01-01'].reset_index(drop=True)

In [ ]:
df

In [ ]:
# df.to_excel(r"E:\A股leverage研究\processed data\data_used_raw.xlsx", index=False)
# df.to_csv(r"E:\A股leverage研究\processed data\data_used_raw.csv", index=False)
df.to_parquet(r"E:\A股leverage研究\processed data\data_used_raw.parquet", index=False)

In [ ]:
df.info()

In [ ]:
df.市场类型.value_counts()